# 02. Data Preprocessing & Feature Engineering

**Dataset:** NYC TLC Green Taxi trip records (2024–2026)  
**Target variable:** `is_tipped` — binary classification (1 if passenger tipped, 0 if not)  
**Scope:** credit-card trips only (payment_type == 1), since cash tips are not recorded

## Sections
0. Imports & Constants  
1. Load Raw Data  
2. Exploratory Data Analysis  
3. Data Cleaning  
4. Define Target Variable  
5. Feature Engineering  
6. Reference / Analysis Split  
7. Save to Hopsworks Feature Store  
8. Save Intermediate Files (for Kedro pipelines)

## 0. Imports & Constants

In [4]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Feature store
import hopsworks

# Paths (relative to notebooks/ folder)
RAW_DATA_PATH = "../data/01_raw/green_taxi"
ZONE_LOOKUP_PATH = "../data/01_raw/taxi_zone_lookup.csv"
INTERMEDIATE_PATH = "../data/02_intermediate"

# Split: 2024 = reference (train), 2025+ = analysis (batch/drift)
REFERENCE_YEARS = [2024]
ANALYSIS_YEARS  = [2025, 2026]

RANDOM_STATE = 42

# Target
TARGET_COL = "is_tipped"

# Hopsworks — fill in your credentials or load from .env
FS_API_KEY      = os.getenv("FS_API_KEY", "YOUR_API_KEY_HERE")
FS_PROJECT_NAME = os.getenv("FS_PROJECT_NAME", "mlops_novaims")

print("Imports OK")

ModuleNotFoundError: No module named 'numpy'

## 1. Load Raw Data

We load all monthly parquet files and concatenate them. The folder structure is:
```
data/01_raw/green_taxi/{year}/green_tripdata_{year}-{month}.parquet
```

In [ ]:
def load_parquet_folder(base_path: str, years: list) -> pd.DataFrame:
    """Load all monthly parquet files for the given years."""
    files = []
    for year in years:
        pattern = os.path.join(base_path, str(year), "*.parquet")
        files.extend(sorted(glob.glob(pattern)))
    
    print(f"Loading {len(files)} parquet files...")
    dfs = [pd.read_parquet(f) for f in files]
    df = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
    return df


# Load reference (2024) and analysis (2025-2026) separately
ref_raw  = load_parquet_folder(RAW_DATA_PATH, REFERENCE_YEARS)
ana_raw  = load_parquet_folder(RAW_DATA_PATH, ANALYSIS_YEARS)

# Combine for EDA
df_all = pd.concat([ref_raw, ana_raw], ignore_index=True)
print(f"\nTotal combined: {len(df_all):,} rows")

In [ ]:
df_all.head(3)

In [ ]:
df_all.dtypes

In [ ]:
# Load zone lookup
zone_lookup = pd.read_csv(ZONE_LOOKUP_PATH)
zone_lookup.head()

## 2. Exploratory Data Analysis

In [ ]:
print("=== Shape ===")
print(df_all.shape)

print("\n=== Null % ===")
null_pct = (df_all.isnull().mean() * 100).round(1)
print(null_pct[null_pct > 0])

In [ ]:
df_all.describe()

In [ ]:
# Payment type distribution
# 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown
payment_labels = {1: 'Credit card', 2: 'Cash', 3: 'No charge', 4: 'Dispute', 5: 'Unknown'}
payment_counts = df_all['payment_type'].map(payment_labels).value_counts()
print("Payment type distribution:")
print(payment_counts)
print(f"\nCredit card share: {payment_counts.get('Credit card', 0) / len(df_all) * 100:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Only credit card trips for tip analysis
cc_df = df_all[df_all['payment_type'] == 1].copy()

axes[0, 0].hist(cc_df['trip_distance'].clip(0, 20), bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Trip Distance (credit card, capped 20mi)')
axes[0, 0].set_xlabel('miles')

axes[0, 1].hist(cc_df['fare_amount'].clip(0, 80), bins=50, color='steelblue', edgecolor='white')
axes[0, 1].set_title('Fare Amount (capped $80)')
axes[0, 1].set_xlabel('USD')

axes[0, 2].hist(cc_df['tip_amount'].clip(0, 20), bins=50, color='coral', edgecolor='white')
axes[0, 2].set_title('Tip Amount (credit card, capped $20)')
axes[0, 2].set_xlabel('USD')

tip_rate = (cc_df['tip_amount'] > 0).mean() * 100
axes[1, 0].pie([tip_rate, 100 - tip_rate], labels=['Tipped', 'Not tipped'],
               colors=['coral', 'lightgrey'], autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('Tipping rate (credit card trips)')

pickup_hours = pd.to_datetime(cc_df['lpep_pickup_datetime']).dt.hour
axes[1, 1].hist(pickup_hours, bins=24, color='steelblue', edgecolor='white')
axes[1, 1].set_title('Pickup Hour Distribution')
axes[1, 1].set_xlabel('Hour of day')

dow = pd.to_datetime(cc_df['lpep_pickup_datetime']).dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow.value_counts().reindex(dow_order).plot(kind='bar', ax=axes[1, 2], color='steelblue')
axes[1, 2].set_title('Trips by Day of Week')
axes[1, 2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Tip rate by hour of day
cc_df2 = cc_df.copy()
cc_df2['hour'] = pd.to_datetime(cc_df2['lpep_pickup_datetime']).dt.hour
cc_df2['is_tipped'] = (cc_df2['tip_amount'] > 0).astype(int)

tip_by_hour = cc_df2.groupby('hour')['is_tipped'].mean() * 100

plt.figure(figsize=(12, 4))
plt.bar(tip_by_hour.index, tip_by_hour.values, color='coral')
plt.axhline(tip_by_hour.mean(), color='black', linestyle='--', label=f'Mean: {tip_by_hour.mean():.1f}%')
plt.xlabel('Hour of Day')
plt.ylabel('Tip Rate (%)')
plt.title('Tip Rate by Hour of Day (credit card trips)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Data Cleaning

Steps:
- Drop `ehail_fee` (100% null)
- Filter to credit-card trips only (`payment_type == 1`)
- Remove rows with invalid trip distances (≤ 0)
- Remove rows with negative fares
- Remove outliers (extreme distances / durations)
- Remove rows with no pickup/dropoff timestamps

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean raw green taxi data."""
    n_start = len(df)
    
    # Drop fully null column
    df = df.drop(columns=['ehail_fee'], errors='ignore')
    
    # Filter to credit card trips only (cash tips not recorded)
    df = df[df['payment_type'] == 1].copy()
    print(f"After credit card filter: {len(df):,} ({len(df)/n_start*100:.1f}% of original)")
    
    # Drop rows missing essential fields
    df = df.dropna(subset=['lpep_pickup_datetime', 'lpep_dropoff_datetime',
                            'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount'])
    
    # Filter valid trip distances (> 0 and < 100 miles)
    df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
    
    # Filter valid fares (> 0 and < 500)
    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
    
    # Filter valid timestamps (no future dates, no dates before 2024)
    df = df[df['lpep_pickup_datetime'] >= '2024-01-01']
    df = df[df['lpep_dropoff_datetime'] > df['lpep_pickup_datetime']]
    
    # Compute duration for outlier removal
    duration_min = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60
    # Keep trips between 1 and 180 minutes
    df = df[(duration_min >= 1) & (duration_min <= 180)]
    
    print(f"After cleaning: {len(df):,} rows remaining")
    return df.reset_index(drop=True)


ref_clean = clean_data(ref_raw)
print()
ana_clean = clean_data(ana_raw)

## 4. Define Target Variable

**`is_tipped`**: 1 if the passenger tipped (tip_amount > 0), 0 otherwise.  
Since we already filtered to credit card trips, this is a valid signal.

> Note: `tip_amount` and `total_amount` are dropped from features to avoid leakage.

In [ ]:
def add_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['is_tipped'] = (df['tip_amount'] > 0).astype(int)
    return df

ref_clean = add_target(ref_clean)
ana_clean = add_target(ana_clean)

print("Reference (2024) target distribution:")
print(ref_clean['is_tipped'].value_counts(normalize=True).round(3))
print()
print("Analysis (2025-2026) target distribution:")
print(ana_clean['is_tipped'].value_counts(normalize=True).round(3))

## 5. Feature Engineering

New features derived from raw columns:

| Feature | Description | Source |
|---|---|---|
| `trip_duration_min` | Trip duration in minutes | pickup/dropoff datetime |
| `pickup_hour` | Hour of pickup (0–23) | lpep_pickup_datetime |
| `pickup_dayofweek` | Day of week (0=Mon, 6=Sun) | lpep_pickup_datetime |
| `pickup_month` | Month (1–12) | lpep_pickup_datetime |
| `is_weekend` | 1 if Saturday or Sunday | pickup_dayofweek |
| `is_rush_hour` | 1 if weekday 7–9am or 5–7pm | pickup_hour + is_weekend |
| `is_night` | 1 if pickup between 10pm–5am | pickup_hour |
| `speed_mph` | Average trip speed (miles/hour) | distance + duration |
| `fare_per_mile` | Fare divided by distance | fare_amount + trip_distance |
| `is_airport` | 1 if PU or DO at JFK/LGA/EWR | PULocationID / DOLocationID |
| `PU_borough` | Borough of pickup zone | zone lookup |
| `DO_borough` | Borough of dropoff zone | zone lookup |

In [ ]:
# NYC airport zone IDs (JFK=132, LGA=138, EWR=1)
AIRPORT_ZONE_IDS = {1, 132, 138}

def engineer_features(df: pd.DataFrame, zone_lookup: pd.DataFrame) -> pd.DataFrame:
    """Engineer features from raw green taxi trip data."""
    df = df.copy()
    
    # --- Temporal features ---
    pickup = pd.to_datetime(df['lpep_pickup_datetime'])
    dropoff = pd.to_datetime(df['lpep_dropoff_datetime'])
    
    df['trip_duration_min'] = (dropoff - pickup).dt.total_seconds() / 60
    df['pickup_hour']       = pickup.dt.hour
    df['pickup_dayofweek']  = pickup.dt.dayofweek  # 0=Monday, 6=Sunday
    df['pickup_month']      = pickup.dt.month
    
    df['is_weekend']   = (df['pickup_dayofweek'] >= 5).astype(int)
    df['is_rush_hour'] = (
        (df['is_weekend'] == 0) &
        (df['pickup_hour'].isin(range(7, 10)) | df['pickup_hour'].isin(range(17, 20)))
    ).astype(int)
    df['is_night'] = (df['pickup_hour'].isin(list(range(22, 24)) + list(range(0, 6)))).astype(int)
    
    # --- Trip efficiency features ---
    df['speed_mph'] = (
        df['trip_distance'] / (df['trip_duration_min'] / 60)
    ).clip(0, 80)  # cap at 80 mph
    
    df['fare_per_mile'] = (df['fare_amount'] / df['trip_distance']).clip(0, 50)
    
    # --- Location features ---
    df['is_airport'] = (
        df['PULocationID'].isin(AIRPORT_ZONE_IDS) |
        df['DOLocationID'].isin(AIRPORT_ZONE_IDS)
    ).astype(int)
    
    # Join borough info from zone lookup
    zone_map = zone_lookup.set_index('LocationID')['Borough'].to_dict()
    df['PU_borough'] = df['PULocationID'].map(zone_map).fillna('Unknown')
    df['DO_borough'] = df['DOLocationID'].map(zone_map).fillna('Unknown')
    
    # Same-borough trip
    df['same_borough'] = (df['PU_borough'] == df['DO_borough']).astype(int)
    
    return df


ref_feat = engineer_features(ref_clean, zone_lookup)
ana_feat = engineer_features(ana_clean, zone_lookup)

print("Reference features shape:", ref_feat.shape)
print("Analysis features shape: ", ana_feat.shape)

In [ ]:
# Preview engineered features
new_cols = ['trip_duration_min', 'pickup_hour', 'pickup_dayofweek', 'pickup_month',
            'is_weekend', 'is_rush_hour', 'is_night', 'speed_mph',
            'fare_per_mile', 'is_airport', 'PU_borough', 'DO_borough', 'same_borough', 'is_tipped']
ref_feat[new_cols].head()

In [ ]:
# Correlation of engineered features with target
numeric_cols = ['trip_duration_min', 'pickup_hour', 'pickup_dayofweek', 'pickup_month',
                'is_weekend', 'is_rush_hour', 'is_night', 'speed_mph',
                'fare_per_mile', 'is_airport', 'same_borough',
                'trip_distance', 'fare_amount', 'passenger_count',
                'is_tipped']

corr = ref_feat[numeric_cols].corr()['is_tipped'].drop('is_tipped').sort_values()

plt.figure(figsize=(8, 6))
corr.plot(kind='barh', color=['coral' if v < 0 else 'steelblue' for v in corr])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with is_tipped')
plt.tight_layout()
plt.show()

In [ ]:
# Tip rate by borough
borough_tip_rate = ref_feat.groupby('PU_borough')['is_tipped'].mean().sort_values(ascending=False) * 100
print("Tip rate by pickup borough (reference data):")
print(borough_tip_rate.round(1))

## 6. Select Final Feature Set

Drop columns that are:
- **Leakage**: `tip_amount`, `total_amount` (contain or derive the target)
- **Raw temporal**: `lpep_pickup_datetime`, `lpep_dropoff_datetime` (replaced by engineered features)
- **Redundant**: `payment_type` (we filtered to credit card only, all == 1)
- **Administrative**: `store_and_fwd_flag`, `VendorID`

In [ ]:
COLS_TO_DROP = [
    'tip_amount',            # leakage (used to create target)
    'total_amount',          # leakage (includes tip)
    'lpep_pickup_datetime',  # replaced by engineered features
    'lpep_dropoff_datetime', # replaced by engineered features
    'payment_type',          # constant after filtering (all == 1)
    'store_and_fwd_flag',    # administrative
    'VendorID',              # administrative
]

def select_features(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_drop = [c for c in COLS_TO_DROP if c in df.columns]
    return df.drop(columns=cols_to_drop)

ref_final = select_features(ref_feat)
ana_final = select_features(ana_feat)

print("Final feature columns:")
print(ref_final.columns.tolist())
print(f"\nReference shape: {ref_final.shape}")
print(f"Analysis shape:  {ana_final.shape}")

In [ ]:
# Final null check
print("Nulls in reference features:")
print(ref_final.isnull().sum()[ref_final.isnull().sum() > 0])

# Fill any remaining nulls
ref_final = ref_final.fillna(-1)
ana_final = ana_final.fillna(-1)

print("\nNull check after fill: OK" if ref_final.isnull().sum().sum() == 0 else "Still has nulls!")

## 7. Save to Hopsworks Feature Store

We create two feature groups:
- **`green_taxi_ref_features`** — reference (training) data from 2024
- **`green_taxi_ana_features`** — analysis (batch/drift) data from 2025–2026

Set `to_feature_store = True` to actually upload. Leave `False` during development.

In [ ]:
TO_FEATURE_STORE = False  # Set to True to upload

if TO_FEATURE_STORE:
    project = hopsworks.login(
        api_key_value=FS_API_KEY,
        project=FS_PROJECT_NAME
    )
    fs = project.get_feature_store()
    print(f"Connected to project: {FS_PROJECT_NAME}")

In [ ]:
def upload_feature_group(
    fs,
    df: pd.DataFrame,
    group_name: str,
    version: int = 1,
    description: str = ""
):
    """Create or get a feature group and upload data."""
    fg = fs.get_or_create_feature_group(
        name=group_name,
        version=version,
        description=description,
        primary_key=['index'],
        online_enabled=False,
    )
    df_upload = df.reset_index().rename(columns={'index': 'index'})
    fg.insert(df_upload, overwrite=False, write_options={"wait_for_job": True})
    fg.compute_statistics()
    print(f"Uploaded {len(df):,} rows to feature group '{group_name}'")
    return fg


if TO_FEATURE_STORE:
    fg_ref = upload_feature_group(
        fs, ref_final,
        group_name="green_taxi_ref_features",
        description="Green Taxi 2024 reference features for training"
    )
    fg_ana = upload_feature_group(
        fs, ana_final,
        group_name="green_taxi_ana_features",
        description="Green Taxi 2025-2026 analysis features for batch inference and drift"
    )
else:
    print("Feature store upload skipped (TO_FEATURE_STORE=False). Set to True to upload.")

## 8. Save Intermediate Files (for Kedro pipelines)

Save the engineered datasets to `data/02_intermediate/` so the Kedro `ingestion` and `split_data` pipelines can pick them up.

In [ ]:
os.makedirs(INTERMEDIATE_PATH, exist_ok=True)

ref_out_path = os.path.join(INTERMEDIATE_PATH, "ref_data.parquet")
ana_out_path = os.path.join(INTERMEDIATE_PATH, "ana_data.parquet")

ref_final.to_parquet(ref_out_path, index=False)
ana_final.to_parquet(ana_out_path, index=False)

print(f"Saved reference data:  {ref_out_path} ({len(ref_final):,} rows)")
print(f"Saved analysis data:   {ana_out_path} ({len(ana_final):,} rows)")

In [ ]:
# Summary
print("=" * 50)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 50)
print(f"Reference (train) rows : {len(ref_final):,}")
print(f"Analysis (batch) rows  : {len(ana_final):,}")
print(f"Number of features     : {len(ref_final.columns) - 1}")
print(f"Target column          : {TARGET_COL}")
print(f"Target balance (ref)   : {ref_final[TARGET_COL].mean():.1%} tipped")
print()
print("Feature columns:")
feature_cols = [c for c in ref_final.columns if c != TARGET_COL]
for c in feature_cols:
    print(f"  - {c}")